In [1]:
# 05_generation.ipynb
# Generation and Trustworthiness Analysis
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
GEN_RESULTS_PATH = PROJECT_ROOT / "data" / "evaluation" / "generation_demo_results.json"
GEN_SUMMARY_PATH = PROJECT_ROOT / "data" / "evaluation" / "generation_summary.json"

In [2]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

generation_results = load_json(GEN_RESULTS_PATH)
generation_summary = load_json(GEN_SUMMARY_PATH)

print("Generated query results:", len(generation_results))
generation_summary

Generated query results: 4


{'queries': 4,
 'high_trust': 1,
 'medium_trust': 0,
 'low_trust': 0,
 'insufficient_evidence': 3,
 'avg_grounding_overlap': 0.283334,
 'avg_trust_score': 0.283334}

In [3]:
#Summary table
pd.DataFrame([generation_summary])

,queries,high_trust,medium_trust,low_trust,insufficient_evidence,avg_grounding_overlap,avg_trust_score
0,4,1,0,0,3,0.283334,0.283334


In [6]:
# Query-level answer summary
rows = []

# Assuming your variable is loaded from generation_demo_results.json
for rec in generation_results:
    analysis_block = rec.get("analysis", {})
    complexity_block = analysis_block.get("complexity", {})
    
    # Safe references to generation and validation blocks
    gen_block = rec.get("generation", {})
    val_block = rec.get("validation", {})
    
    rows.append({
        "query": rec.get("query"),
        "retrieval_mode": rec.get("retrieval_mode"),
        # ✅ FIX: Fetch from nested complexity block or use fallback values
        "complexity_label": complexity_block.get("complexity_label", "low"),
        "complexity_score": complexity_block.get("complexity_score", 0.0),
        
        # ✅ FIX: Defensive lookups for generation details
        "parsed_successfully": gen_block.get("parsed_successfully", True),
        "model_confidence": gen_block.get("confidence", None),
        "insufficient_evidence": gen_block.get("insufficient_evidence", None),
        
        # ✅ FIX: Defensive lookups for validation details
        "citation_count": val_block.get("citation_count", 0),
        "grounding_overlap": val_block.get("grounding_overlap", 0.0),
        "trust_score": val_block.get("trust_score", 0.0),
        "trust_label": val_block.get("trust_label", "unknown"),
    })

answer_df = pd.DataFrame(rows)
answer_df

,query,retrieval_mode,complexity_label,complexity_score,parsed_successfully,model_confidence,insufficient_evidence,citation_count,grounding_overlap,trust_score,trust_label
0,ሲቢኢ ብር እንዴት እመዝገባለሁ?,hybrid,low,0.2398,True,low,True,0,0.166667,0.166667,insufficient_evidence
1,የቁጠባ ሂሳብ ለመክፈት ምን ያስፈልጋል?,hybrid,low,0.2420,True,low,True,0,0.166667,0.166667,insufficient_evidence
2,የATM ካርድ PIN ከረሳሁ ምን ላድርግ?,sparse,low,0.2774,True,low,True,0,0.000000,0.000000,insufficient_evidence
3,የሞባይል ባንኪንግ ኮሚሽን ስንት ነው?,hybrid,low,0.2379,True,high,False,2,0.800000,0.800000,high


In [7]:
#Trust label distribution
answer_df["trust_label"].value_counts()

trust_label
insufficient_evidence    3
high                     1
Name: count, dtype: int64

In [8]:
#Average trust by retrieval mode
answer_df.groupby("retrieval_mode")[["grounding_overlap", "trust_score"]].mean().reset_index()

,retrieval_mode,grounding_overlap,trust_score
0,hybrid,0.377778,0.377778
1,sparse,0.000000,0.000000


In [10]:
# Inspect one full example
idx = 0
sample = generation_results[idx]

# Extract nested dictionaries safely
analysis_block = sample.get("analysis", {})
complexity_block = analysis_block.get("complexity", {})
gen_block = sample.get("generation", {})
val_block = sample.get("validation", {})

print("================================================================================")
print("🔬 FULL PIPELINE SAMPLE INSPECTION (INDEX {})".format(idx))
print("================================================================================")
print("QUERY:")
print(sample.get("query"))

print("\nRETRIEVAL MODE:", sample.get("retrieval_mode"))
print("COMPLEXITY:    {} ({:.4f})".format(
    complexity_block.get("complexity_label", "unknown"), 
    complexity_block.get("complexity_score", 0.0)
))

print("\n--------------------------------------------------------------------------------")
print("📝 GENERATED ANSWER:")
print(gen_block.get("answer", "No answer generated."))

print("\n📌 CITATIONS / SOURCES USED:")
citations = gen_block.get("citations", [])
if isinstance(citations, list) and len(citations) > 0:
    for i, cit in enumerate(citations, start=1):
        print(f"  [{i}] {cit}")
else:
    print("  No text sources explicitly cited by the model.")

print("\n🛡️ VALIDATION ENGINE METRICS:")
if val_block:
    for metric, value in val_block.items():
        print(f"  {metric:<25}: {value}")
else:
    print("  No validation telemetry recorded for this query.")
print("================================================================================")

🔬 FULL PIPELINE SAMPLE INSPECTION (INDEX 0)
QUERY:
ሲቢኢ ብር እንዴት እመዝገባለሁ?

RETRIEVAL MODE: hybrid
COMPLEXITY:    low (0.2398)

--------------------------------------------------------------------------------
📝 GENERATED ANSWER:
በቀረበው ማስረጃ መሰረት በቂ መረጃ አልተገኘም።

📌 CITATIONS / SOURCES USED:
  No text sources explicitly cited by the model.

🛡️ VALIDATION ENGINE METRICS:
  citation_count           : 0
  citation_ok              : True
  grounding_overlap        : 0.166667
  unsupported_penalty      : 0.0
  trust_score              : 0.166667
  trust_label              : insufficient_evidence


In [11]:
#Show retrieved evidence used by one answer
for i, item in enumerate(sample["retrieved_chunks"][:3], start=1):
    print("=" * 80)
    print(f"RANK {i}")
    print("chunk_id:", item["chunk_id"])
    print("score:", item["score"])
    print("source_file:", item["metadata"]["source_file"])
    print("-" * 80)
    print(item["text"][:1500])
    print()

RANK 1
chunk_id: chunk_000138
score: 0.6125
source_file: cleaned28_Press_Release_on_IDFR_307ddd5d1c_c5a16feb0d.txt
--------------------------------------------------------------------------------
ዙሮች ለተካሄዱ መርሃ-ግብሮች ከ46.2 ሚሊዮን ብር በላይ ግምት ያላቸው ሽልማቶችን ለደንበኞች በመሸለም ትልቅ የፕሮሞሽን ስራ ተከናውኗል፡፡ ➢ የጥቁር ገበያውን ተፅእኖ ለመቀነስ በሚደረገው ሀገር አቀፍ ጥረት ውስጥ የመፍትሄው አካል ለመሆን እየተሰራ ይገኛል፡፡ በአለም አቀፍ ደረጃ ጁን 16/ሰኔ 9 ቀን የሚከበረውን አለም አቀፍ የሀዋላ ቀንን ባንካችን በተለየ ሁኔታ እንዲከበር ወስኗል፡፡ ቀኑ እንዲዘከር ያስፈለገበት ዋነኛው ምክንያት በሀገር ልማት ውስጥ ወሳኝ ቦታ ያለው የውጪ ሃገር ገንዘብ ግኝታችን በመደበኛው የባንክ ስርአት ውስጥ እንዲያልፍ መልእከት የማስተላለፊያ መድረክ መፍጠር ነው፡፡ ህብረተሰባችንን የማንቃት ተግባር በአግባቡ ከተከናወነ በርካታ ታዳጊ ሃገራት በዘርፉ ከደረሱበት የተጠቃሚነት ደረጃ መድረስ እንደሚቻል የኢትዮጵያ ንግድ ባንክ በፅኑ ያምናል፡፡ 3 ገ ፅ

RANK 2
chunk_id: chunk_000098
score: 0.558996
source_file: cleaned28_CBE_Noor_Brand_flyer_2_f0213d1a4f_922a1170db (1).txt
--------------------------------------------------------------------------------
ሰ የገንዘብ ማስተላለፍ ሀዋላ አገልግሎት ደንበኞች ከአገር ውስጥ እና ከአገር ውጪ በባንኩ ቅርንጫፎች በኩል የተላከላቸውን ገንዘብ በአቅራቢያቸው ከሚገኝ ማንኛውም የሲቢኢ ኑ

In [12]:
#Paper-ready answer examples table
paper_examples = []
for rec in generation_results:
    paper_examples.append({
        "Query": rec["query"],
        "Retrieval mode": rec["retrieval_mode"],
        "Answer": rec["generation"]["answer"][:250],
        "Citations": ", ".join(rec["generation"]["citations"]),
        "Trust label": rec["validation"]["trust_label"],
        "Grounding overlap": rec["validation"]["grounding_overlap"],
    })

paper_examples_df = pd.DataFrame(paper_examples)
paper_examples_df

,Query,Retrieval mode,Answer,Citations,Trust label,Grounding overlap
0,ሲቢኢ ብር እንዴት እመዝገባለሁ?,hybrid,በቀረበው ማስረጃ መሰረት በቂ መረጃ አልተገኘም።,,insufficient_evidence,0.166667
1,የቁጠባ ሂሳብ ለመክፈት ምን ያስፈልጋል?,hybrid,በቀረበው ማስረጃ መሰረት በቂ መረጃ አልተገኘም።,,insufficient_evidence,0.166667
2,የATM ካርድ PIN ከረሳሁ ምን ላድርግ?,sparse,በቀረበው ማስረጃ መሰረት በቂ መረጃ አልተገኘም።,,insufficient_evidence,0.000000
3,የሞባይል ባንኪንግ ኮሚሽን ስንት ነው?,hybrid,የሞባይል ባንኪንግ ነው የሚያስችል የሚገበራ የባንክ አገልግሎቶችን እንዲያ...,"chunk_000001, chunk_000113",high,0.800000


In [13]:
#Candidate table for paper
paper_table = answer_df[[
    "query",
    "retrieval_mode",
    "model_confidence",
    "citation_count",
    "grounding_overlap",
    "trust_score",
    "trust_label"
]]
paper_table

,query,retrieval_mode,model_confidence,citation_count,grounding_overlap,trust_score,trust_label
0,ሲቢኢ ብር እንዴት እመዝገባለሁ?,hybrid,low,0,0.166667,0.166667,insufficient_evidence
1,የቁጠባ ሂሳብ ለመክፈት ምን ያስፈልጋል?,hybrid,low,0,0.166667,0.166667,insufficient_evidence
2,የATM ካርድ PIN ከረሳሁ ምን ላድርግ?,sparse,low,0,0.000000,0.000000,insufficient_evidence
3,የሞባይል ባንኪንግ ኮሚሽን ስንት ነው?,hybrid,high,2,0.800000,0.800000,high
